# Example 02 — Two-DOF Chain with Cubic Spring

Frequency-response analysis of a 2-DOF chain of oscillators with a cubic spring nonlinearity at DOF 0, showing resonance peaks for both degrees of freedom. (Krack & Gross 2019, §5)

In [ ]:
# --- Imports ---
import sys
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import matplotlib
matplotlib.use('Agg')

_repo_root = Path('..').resolve()
if str(_repo_root / 'src') not in sys.path:
    sys.path.insert(0, str(_repo_root / 'src'))

from nlvib.systems.oscillators import ChainOfOscillators
from nlvib.nonlinearities.elements import cubic_spring
from nlvib.solvers.harmonic_balance import hb_residual
from nlvib.continuation.solver import ContinuationSolver, ContinuationOptions
from nlvib.visualization.plots import plot_frf, plot_harmonic_content

In [ ]:
# --- System setup ---
MASSES = [1.0, 1.0]
STIFFNESSES = [1.0, 0.5, 1.0]
DAMPINGS = [0.01, 0.01, 0.01]
CUBIC_K3 = 1.0
EXCITATION_AMPLITUDE = 0.05
N_HARMONICS = 3
OMEGA_START = 0.3
OMEGA_END = 1.5

system = ChainOfOscillators(masses=MASSES, stiffnesses=STIFFNESSES, dampings=DAMPINGS)
system.add_nonlinear_element(cubic_spring(k3=CUBIC_K3, dof_index=0))
n_dof = system.n_dof
n_total = n_dof * (2 * N_HARMONICS + 1)
print(f'System: {system}')
print(f'n_dof={n_dof}, n_harmonics={N_HARMONICS}, n_total_HB={n_total}')

In [ ]:
# --- Run HB continuation ---
excitation = {'dof': 0, 'amplitude': EXCITATION_AMPLITUDE}
Q0 = np.zeros(n_total)
omega0 = OMEGA_START
for _ in range(50):
    R, J = hb_residual(Q0, omega0, system, N_HARMONICS, excitation)
    if np.linalg.norm(R) < 1e-10: break
    try: dQ = np.linalg.solve(J, -R)
    except np.linalg.LinAlgError: dQ = np.linalg.lstsq(J, -R, rcond=None)[0]
    Q0 = Q0 + dQ

def residual_fn(x, lam):
    return hb_residual(x, lam, system, N_HARMONICS, excitation)

opts = ContinuationOptions(
    ds_initial=0.02, ds_min=1e-5, ds_max=0.1, max_steps=800,
    max_newton_iter=25, newton_tol=1e-8, adapt_step=True,
    lambda_min=OMEGA_START - 0.05, lambda_max=OMEGA_END + 0.05,
)
result = ContinuationSolver().run(residual_fn, Q0, omega0, opts)
print(f'Continuation: {result.n_steps} steps, {result.message}')

In [ ]:
# --- Extract FRF and save plots ---
@dataclass
class FRFResult:
    omega: np.ndarray
    amplitude: np.ndarray
    stability: np.ndarray

solutions = result.solutions
omega_arr = solutions[:, -1]

amplitude = np.zeros((n_dof, len(omega_arr)))
for i_step in range(len(omega_arr)):
    Q = solutions[i_step, :-1]
    for i_dof in range(n_dof):
        c_idx = (2*1-1)*n_dof + i_dof
        s_idx = 2*1*n_dof + i_dof
        amplitude[i_dof, i_step] = np.sqrt(Q[c_idx]**2 + Q[s_idx]**2)

mask = (omega_arr >= OMEGA_START) & (omega_arr <= OMEGA_END)
frf_plot = FRFResult(
    omega=omega_arr[mask],
    amplitude=amplitude[:, mask],
    stability=~result.stability[mask],
)

output_dir = Path('..') / 'examples' / '02_two_dof_cubic' / 'output'
output_dir.mkdir(parents=True, exist_ok=True)

fig0 = plot_frf(frf_plot, dof=0, harmonic=1)
fig0.suptitle('Two-DOF Cubic Spring — FRF at DOF 0', fontsize=11)
fig0.savefig(output_dir / 'frf_dof0.png', dpi=150, bbox_inches='tight')
print('Saved frf_dof0.png')

fig1 = plot_frf(frf_plot, dof=1, harmonic=1)
fig1.suptitle('Two-DOF Cubic Spring — FRF at DOF 1', fontsize=11)
fig1.savefig(output_dir / 'frf_dof1.png', dpi=150, bbox_inches='tight')
print('Saved frf_dof1.png')

# Harmonic content at peak DOF 0
amp_fund0 = np.array([np.sqrt(solutions[i, n_dof]**2 + solutions[i, 2*n_dof]**2) for i in range(len(omega_arr))])
peak_idx = int(np.argmax(amp_fund0))
Q_peak = solutions[peak_idx, :-1]
omega_at_peak = float(solutions[peak_idx, -1])
Q_harmonics = np.array([np.sqrt(Q_peak[(2*h-1)*n_dof]**2 + Q_peak[2*h*n_dof]**2) for h in range(1, N_HARMONICS+1)])

fig_hc = plot_harmonic_content(Q_harmonics, omega_at_peak)
fig_hc.suptitle('Harmonic Content at Peak (DOF 0)', fontsize=11)
fig_hc.savefig(output_dir / 'harmonic_content.png', dpi=150, bbox_inches='tight')
print('Saved harmonic_content.png')

In [ ]:
# --- Display FRF DOF 0 inline ---
fig0

In [ ]:
# --- Display FRF DOF 1 inline ---
fig1

In [ ]:
# --- Display harmonic content inline ---
fig_hc

In [ ]:
# --- Summary ---
amp_dof0 = frf_plot.amplitude[0, :]
amp_dof1 = frf_plot.amplitude[1, :]
peak_amp_dof0 = float(np.max(amp_dof0))
peak_amp_dof1 = float(np.max(amp_dof1))
omega_peak_dof0 = float(frf_plot.omega[np.argmax(amp_dof0)])
omega_peak_dof1 = float(frf_plot.omega[np.argmax(amp_dof1)])
print('=' * 55)
print('SUMMARY — Example 02: Two-DOF Cubic Spring')
print('=' * 55)
print(f'  Peak amplitude DOF 0 : {peak_amp_dof0:.6f}  at omega = {omega_peak_dof0:.4f} rad/s')
print(f'  Peak amplitude DOF 1 : {peak_amp_dof1:.6f}  at omega = {omega_peak_dof1:.4f} rad/s')
print(f'  Harmonic content at peak (DOF 0):')
for h, amp_h in enumerate(Q_harmonics, start=1):
    print(f'    H{h}: {amp_h:.6f}')
print('=' * 55)